In [ ]:
import os, sys, json, subprocess
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
PLAIN_URL = UserSecretsClient().get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (out.stdout + out.stderr).replace(TOKEN, "***")
    print(text)
    if out.returncode != 0:
        raise RuntimeError(text)

def push(msg, notebook=None):
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks sft.py submission_adapter.py && (git diff --cached --quiet || '
        f'git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')
    print("✅ Pushed to GitHub")

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL}")          # keep the token out of saved files
run(f"git pull -q {REPO_URL} main")
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run("pip uninstall -y -q torchao || true")
run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" accelerate pytest')

In [ ]:
get_ipython().run_cell_magic('writefile', 'sft.py', 'import torch, torch.nn.functional as F\n\nPROMPT_TEMPLATE = "### Instruction:\\n{instruction}\\n\\n### Response:\\n"\n\ndef format_prompt(instruction):\n    return PROMPT_TEMPLATE.format(instruction=instruction)\n\ndef build_sft_batch(tokenizer, prompts, responses, max_length):\n    """prompt + response + EOS, truncate, right-pad. Labels: -100 on prompt and padding."""\n    eos = [] if tokenizer.eos_token_id is None else [tokenizer.eos_token_id]\n    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (eos[0] if eos else 0)\n    rows = []\n    for p, r in zip(prompts, responses):\n        p_ids = tokenizer.encode(p, add_special_tokens=False)\n        r_ids = tokenizer.encode(r, add_special_tokens=False) + eos        # EOS before truncation\n        rows.append(((p_ids + r_ids)[:max_length], ([-100] * len(p_ids) + r_ids)[:max_length]))\n    width = max(len(ids) for ids, _ in rows)\n    return {\n        "input_ids": torch.tensor([ids + [pad] * (width - len(ids)) for ids, _ in rows]),\n        "attention_mask": torch.tensor([[1] * len(ids) + [0] * (width - len(ids)) for ids, _ in rows]),\n        "labels": torch.tensor([lab + [-100] * (width - len(lab)) for _, lab in rows]),\n    }\n\ndef sequence_logprobs(model, batch):\n    """Per row: summed log-prob of response tokens, and number of response tokens."""\n    batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}\n    logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits[:, :-1].float()\n    labels = batch["labels"][:, 1:]                                         # token t predicts t+1\n    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),\n                          ignore_index=-100, reduction="none").view(labels.shape)\n    mask = labels.ne(-100)\n    return -(nll * mask).sum(1), mask.sum(1)\n\ndef compute_response_logprobs(model, tokenizer, prompts, responses, max_length):\n    return sequence_logprobs(model, build_sft_batch(tokenizer, prompts, responses, max_length))[0]\n\ndef compute_sft_loss(model, tokenizer, prompts, responses, max_length):\n    """Mean negative log-prob over all response tokens in the batch."""\n    logps, counts = sequence_logprobs(model, build_sft_batch(tokenizer, prompts, responses, max_length))\n    return -logps.sum() / counts.sum().clamp_min(1)\n')

In [ ]:
push("Task 3: SFT code")

In [ ]:
import math, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sft import format_prompt, build_sft_batch, sequence_logprobs

DEBUG = True                               # True = tiny test run (~10 min)
N_TRAIN, N_VAL = (64, 16) if DEBUG else (30000, 1000)
MAX_LEN, BS, ACCUM, LR, WD, CLIP = 512, 2, (2 if DEBUG else 16), 2e-5, 0.1, 1.0
EVAL_EVERY = 2 if DEBUG else 100
OUT_DIR, LOG_DIR = "/kaggle/working/outputs/sft", "results/task3_sft"
os.makedirs(LOG_DIR, exist_ok=True)

# Data: UltraChat, first user/assistant turn, prompt <= 256 tokens, shuffle seed 42
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
rows = []
for ex in load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=42):
    m = ex["messages"]
    if len(m) >= 2 and m[0]["role"] == "user" and m[1]["role"] == "assistant" and m[1]["content"].strip():
        prompt = format_prompt(m[0]["content"].strip())
        if len(tok.encode(prompt)) <= 256:
            rows.append((prompt, m[1]["content"].strip()))
    if len(rows) == N_TRAIN + N_VAL:
        break
val, train = rows[:N_VAL], rows[N_VAL:]

# Model in fp32, AdamW, cosine schedule with 3% warmup, GradScaler for fp16
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", dtype=torch.float32).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
total = len(train) // BS // ACCUM                       # optimizer steps for 1 epoch
warm = max(1, int(0.03 * total))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler("cuda")

def loss_on(pairs):
    with torch.autocast("cuda", dtype=torch.float16):
        logps, n = sequence_logprobs(model, build_sft_batch(tok, [p for p, _ in pairs], [r for _, r in pairs], MAX_LEN))
    return -logps.sum() / n.sum().clamp_min(1)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = [loss_on(val[i:i + 4]).item() for i in range(0, len(val), 4)]
    model.train()
    return sum(losses) / len(losses)

def log(rec):
    print(rec)
    with open(f"{LOG_DIR}/metrics.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")

def save(name):
    model.save_pretrained(f"{OUT_DIR}/{name}"); tok.save_pretrained(f"{OUT_DIR}/{name}")

# Training loop (1 epoch)
open(f"{LOG_DIR}/metrics.jsonl", "w").close()
log({"step": 0, "val_loss": val_loss()})
best, step, running = float("inf"), 0, []
random.seed(42); random.shuffle(train)
model.train()
for i in range(total * ACCUM):
    loss = loss_on(train[i * BS:(i + 1) * BS])
    scaler.scale(loss / ACCUM).backward()                       # gradient accumulation
    running.append(loss.item())
    if (i + 1) % ACCUM:
        continue
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)    # gradient clipping
    scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
    step += 1
    if step % 10 == 0 or DEBUG:
        log({"step": step, "train_loss": sum(running) / len(running), "lr": sched.get_last_lr()[0]}); running = []
    if step % EVAL_EVERY == 0 or step == total:
        v = val_loss()
        log({"step": step, "val_loss": v})
        if v < best:
            best = v; save("best")                              # best checkpoint
        if not DEBUG:
            try: push(f"Task 3: SFT step {step}")               # metrics safe on GitHub
            except Exception as e: print("push failed, continuing:", e)
save("final")                                                   # final checkpoint
print("Best validation loss:", best)

In [ ]:
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in open(f"{LOG_DIR}/metrics.jsonl")]
tr = [r for r in recs if "train_loss" in r]; va = [r for r in recs if "val_loss" in r]
plt.plot([r["step"] for r in tr], [r["train_loss"] for r in tr], label="train")
plt.plot([r["step"] for r in va], [r["val_loss"] for r in va], "o-", label="validation")
plt.xlabel("optimizer step"); plt.ylabel("loss per response token"); plt.title("SFT Qwen2.5-0.5B"); plt.legend()
plt.savefig(f"{LOG_DIR}/sft_loss.png", dpi=150); plt.show()

# Quick check of the best model on 3 AlpacaEval questions
best_model = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/best", dtype=torch.float16).cuda().eval()
for q in json.load(open("eval_sets/alpaca_eval_300_seed42.json"))[:3]:
    enc = tok(format_prompt(q["instruction"]), return_tensors="pt").to("cuda")
    out = best_model.generate(**enc, do_sample=False, max_new_tokens=256, repetition_penalty=1.1,
                              pad_token_id=tok.pad_token_id)
    print("=" * 80, "\nQ:", q["instruction"], "\nA:", tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)[:400])

json.dump({"data": "ultrachat_200k first turn, prompt<=256 tokens, seed 42", "n_train": len(train), "n_val": len(val),
           "epochs": 1, "max_len": MAX_LEN, "batch": BS, "accum": ACCUM, "lr": LR, "weight_decay": WD, "clip": CLIP,
           "warmup": "3%", "schedule": "cosine", "precision": "fp32 weights + fp16 autocast + GradScaler",
           "loss": "mean over response tokens", "best_val_loss": best}, open(f"{LOG_DIR}/config.json", "w"), indent=2)

In [ ]:
from huggingface_hub import HfApi

if not DEBUG:
    # Upload best + final checkpoints to a private Hugging Face repo
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    HF_REPO = f"{api.whoami()['name']}/qwen2.5-0.5b-sft"
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    for name in ["best", "final"]:
        api.upload_folder(folder_path=f"{OUT_DIR}/{name}", path_in_repo=name, repo_id=HF_REPO,
                          commit_message=f"SFT {name} checkpoint")
        print("Uploaded:", name)

    # Save where the models are, so later notebooks know where to load them from
    json.dump({"hf_repo": HF_REPO, "best": "best", "final": "final"},
              open(f"{LOG_DIR}/model_location.json", "w"), indent=2)

    # Push results + this notebook to GitHub
    push("Task 3: SFT finished", notebook="25280065_03_sft.ipynb")

In [ ]:
import os, sys, json, subprocess
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
PLAIN_URL = UserSecretsClient().get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (out.stdout + out.stderr).replace(TOKEN, "***")
    print(text)
    if out.returncode != 0:
        raise RuntimeError(text)

def push(msg, notebook=None):
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks sft.py submission_adapter.py && (git diff --cached --quiet || '
        f'git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')
    print("✅ Pushed to GitHub")

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL}")          # keep the token out of saved files
run(f"git pull -q {REPO_URL} main")
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run("pip uninstall -y -q torchao || true")
run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" accelerate pytest')

In [ ]:
get_ipython().run_cell_magic('writefile', 'sft.py', 'import torch, torch.nn.functional as F\n\nPROMPT_TEMPLATE = "### Instruction:\\n{instruction}\\n\\n### Response:\\n"\n\ndef format_prompt(instruction):\n    return PROMPT_TEMPLATE.format(instruction=instruction)\n\ndef build_sft_batch(tokenizer, prompts, responses, max_length):\n    """prompt + response + EOS, truncate, right-pad. Labels: -100 on prompt and padding."""\n    eos = [] if tokenizer.eos_token_id is None else [tokenizer.eos_token_id]\n    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (eos[0] if eos else 0)\n    rows = []\n    for p, r in zip(prompts, responses):\n        p_ids = tokenizer.encode(p, add_special_tokens=False)\n        r_ids = tokenizer.encode(r, add_special_tokens=False) + eos        # EOS before truncation\n        rows.append(((p_ids + r_ids)[:max_length], ([-100] * len(p_ids) + r_ids)[:max_length]))\n    width = max(len(ids) for ids, _ in rows)\n    return {\n        "input_ids": torch.tensor([ids + [pad] * (width - len(ids)) for ids, _ in rows]),\n        "attention_mask": torch.tensor([[1] * len(ids) + [0] * (width - len(ids)) for ids, _ in rows]),\n        "labels": torch.tensor([lab + [-100] * (width - len(lab)) for _, lab in rows]),\n    }\n\ndef sequence_logprobs(model, batch):\n    """Per row: summed log-prob of response tokens, and number of response tokens."""\n    batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}\n    logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits[:, :-1].float()\n    labels = batch["labels"][:, 1:]                                         # token t predicts t+1\n    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),\n                          ignore_index=-100, reduction="none").view(labels.shape)\n    mask = labels.ne(-100)\n    return -(nll * mask).sum(1), mask.sum(1)\n\ndef compute_response_logprobs(model, tokenizer, prompts, responses, max_length):\n    return sequence_logprobs(model, build_sft_batch(tokenizer, prompts, responses, max_length))[0]\n\ndef compute_sft_loss(model, tokenizer, prompts, responses, max_length):\n    """Mean negative log-prob over all response tokens in the batch."""\n    logps, counts = sequence_logprobs(model, build_sft_batch(tokenizer, prompts, responses, max_length))\n    return -logps.sum() / counts.sum().clamp_min(1)\n')

In [ ]:
push("Task 3: SFT code")

In [ ]:
import math, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sft import format_prompt, build_sft_batch, sequence_logprobs

DEBUG = False                               # True = tiny test run (~10 min)
N_TRAIN, N_VAL = (64, 16) if DEBUG else (30000, 1000)
MAX_LEN, BS, ACCUM, LR, WD, CLIP = 512, 2, (2 if DEBUG else 16), 2e-5, 0.1, 1.0
EVAL_EVERY = 2 if DEBUG else 100
OUT_DIR, LOG_DIR = "/kaggle/working/outputs/sft", "results/task3_sft"
os.makedirs(LOG_DIR, exist_ok=True)

# Data: UltraChat, first user/assistant turn, prompt <= 256 tokens, shuffle seed 42
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
rows = []
for ex in load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=42):
    m = ex["messages"]
    if len(m) >= 2 and m[0]["role"] == "user" and m[1]["role"] == "assistant" and m[1]["content"].strip():
        prompt = format_prompt(m[0]["content"].strip())
        if len(tok.encode(prompt)) <= 256:
            rows.append((prompt, m[1]["content"].strip()))
    if len(rows) == N_TRAIN + N_VAL:
        break
val, train = rows[:N_VAL], rows[N_VAL:]

# Model in fp32, AdamW, cosine schedule with 3% warmup, GradScaler for fp16
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", dtype=torch.float32).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
total = len(train) // BS // ACCUM                       # optimizer steps for 1 epoch
warm = max(1, int(0.03 * total))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler("cuda")

def loss_on(pairs):
    with torch.autocast("cuda", dtype=torch.float16):
        logps, n = sequence_logprobs(model, build_sft_batch(tok, [p for p, _ in pairs], [r for _, r in pairs], MAX_LEN))
    return -logps.sum() / n.sum().clamp_min(1)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = [loss_on(val[i:i + 4]).item() for i in range(0, len(val), 4)]
    model.train()
    return sum(losses) / len(losses)

def log(rec):
    print(rec)
    with open(f"{LOG_DIR}/metrics.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")

def save(name):
    model.save_pretrained(f"{OUT_DIR}/{name}"); tok.save_pretrained(f"{OUT_DIR}/{name}")

# Training loop (1 epoch)
open(f"{LOG_DIR}/metrics.jsonl", "w").close()
log({"step": 0, "val_loss": val_loss()})
best, step, running = float("inf"), 0, []
random.seed(42); random.shuffle(train)
model.train()
for i in range(total * ACCUM):
    loss = loss_on(train[i * BS:(i + 1) * BS])
    scaler.scale(loss / ACCUM).backward()                       # gradient accumulation
    running.append(loss.item())
    if (i + 1) % ACCUM:
        continue
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)    # gradient clipping
    scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
    step += 1
    if step % 10 == 0 or DEBUG:
        log({"step": step, "train_loss": sum(running) / len(running), "lr": sched.get_last_lr()[0]}); running = []
    if step % EVAL_EVERY == 0 or step == total:
        v = val_loss()
        log({"step": step, "val_loss": v})
        if v < best:
            best = v; save("best")                              # best checkpoint
        if not DEBUG:
            try: push(f"Task 3: SFT step {step}")               # metrics safe on GitHub
            except Exception as e: print("push failed, continuing:", e)
save("final")                                                   # final checkpoint
print("Best validation loss:", best)

In [ ]:
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in open(f"{LOG_DIR}/metrics.jsonl")]
tr = [r for r in recs if "train_loss" in r]; va = [r for r in recs if "val_loss" in r]
plt.plot([r["step"] for r in tr], [r["train_loss"] for r in tr], label="train")
plt.plot([r["step"] for r in va], [r["val_loss"] for r in va], "o-", label="validation")
plt.xlabel("optimizer step"); plt.ylabel("loss per response token"); plt.title("SFT Qwen2.5-0.5B"); plt.legend()
plt.savefig(f"{LOG_DIR}/sft_loss.png", dpi=150); plt.show()

# Quick check of the best model on 3 AlpacaEval questions
best_model = AutoModelForCausalLM.from_pretrained(f"{OUT_DIR}/best", dtype=torch.float16).cuda().eval()
for q in json.load(open("eval_sets/alpaca_eval_300_seed42.json"))[:3]:
    enc = tok(format_prompt(q["instruction"]), return_tensors="pt").to("cuda")
    out = best_model.generate(**enc, do_sample=False, max_new_tokens=256, repetition_penalty=1.1,
                              pad_token_id=tok.pad_token_id)
    print("=" * 80, "\nQ:", q["instruction"], "\nA:", tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)[:400])

json.dump({"data": "ultrachat_200k first turn, prompt<=256 tokens, seed 42", "n_train": len(train), "n_val": len(val),
           "epochs": 1, "max_len": MAX_LEN, "batch": BS, "accum": ACCUM, "lr": LR, "weight_decay": WD, "clip": CLIP,
           "warmup": "3%", "schedule": "cosine", "precision": "fp32 weights + fp16 autocast + GradScaler",
           "loss": "mean over response tokens", "best_val_loss": best}, open(f"{LOG_DIR}/config.json", "w"), indent=2)

In [ ]:
from huggingface_hub import HfApi

if not DEBUG:
    # Upload best + final checkpoints to a private Hugging Face repo
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    HF_REPO = f"{api.whoami()['name']}/qwen2.5-0.5b-sft"
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    for name in ["best", "final"]:
        api.upload_folder(folder_path=f"{OUT_DIR}/{name}", path_in_repo=name, repo_id=HF_REPO,
                          commit_message=f"SFT {name} checkpoint")
        print("Uploaded:", name)

    # Save where the models are, so later notebooks know where to load them from
    json.dump({"hf_repo": HF_REPO, "best": "best", "final": "final"},
              open(f"{LOG_DIR}/model_location.json", "w"), indent=2)

    # Push results + this notebook to GitHub
    push("Task 3: SFT finished", notebook="25280065_03_sft.ipynb")